In [ ]:
## NEED TO SPECIFY WHICH VERSION OF PYTHON TO USE 3.15.1

# What we are building

TBC

Add learning objectives to each area to demonstrate

## 1. Setup & Preflight
Install Dependancies (from requirements.txt) <br>
Import <br>
Setup Logging <br>
Config <br>
Preflight Check <br>

### 1.1 Install Depedancies

In [ ]:
# Install dependencies from requirements.txt
%pip install --quiet -r requirements.txt

In [ ]:
# Automate a requirements.txt file using the notebook and versions

from importlib.metadata import distributions

# Write the installed packages and versions to requirements.txt
with open("requirements.txt", "w", encoding="utf-8") as f:
    for distribution in sorted(
        distributions(),
        key=lambda dist: (dist.metadata.get("Name") or "").lower()
    ):
        name = distribution.metadata.get("Name")
        if name:
            f.write(f"{name}=={distribution.version}\n")

In [ ]:
# Import libraries

from urllib.parse import urlparse
import requests
import pandas as pd
from bs4 import BeautifulSoup
import xml.etree.ElementTree as ET
import time
import json
import logging
import requests
from urllib.robotparser import RobotFileParser
from pathlib import Path
import nltk
from urllib.parse import urlencode, urlparse, parse_qs
import hashlib
import os
import re
import secrets
from datetime import datetime, timezone
import spacy





### 1.2 Setup Logging

In [ ]:
# Set up logging and log to both console and file

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s.%(funcName)s: %(message)s",
    handlers=[
        logging.FileHandler("scraper.log"),
        logging.StreamHandler(),
    ],
    force=True)

logger = logging.getLogger("mse_dwp_analysis")
logging.info("Logging started.")

### 1.3 Configuration

In [ ]:
### SCRAPER CONFIGURATION ###
BASE_URL = "https://forums.moneysavingexpert.com/"
DELAY = 2
HEADER = {"User-Agent": "BPP-DataScience-Bot/1.0 (educational project)"}

### FORUM CONFIGURATION ###
FORUM_URL = "https://forums.moneysavingexpert.com/categories/benefits-tax-credits"
NUMBER_OF_THREADS_TO_SCRAPE = 40

### ALETRNATIVE SCRAPING OPTIONS TO CHECK FOR ROBOTS.TXT ###
SITEMAP_URL = "https://forums.moneysavingexpert.com/sitemap-category-benefits-tax-credits-1-1000.xml"
SEARCH_URL = "https://forums.moneysavingexpert.com/search?domain=all_content&query=%22pension%22&scope=site&source=community"

### CACHING CONFIGURATION ###
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
CACHE_PATH = DATA_DIR / "mse_posts.jsonl"
CACHE_META = DATA_DIR / "mse_posts.meta.json"
REFRESH_CACHE = False         # set True to collect afresh from the website, False to use cached data if available


### 1.4 Pre-flight check

Check the environment to ensure XYZ <br>
Add anything to this which requires libraries and access

Source: BPP University (2026) ‘Consolidation 1: Blue Sky Weekly Digest.ipynb’, Unstructured Data. Available at: BPP (Accessed: 16 November 2026).

In [ ]:
def preflight() -> dict:
    """Check what this environment can do, and add to logging.
    If there is no network access and no cache, raise an error and log a critical message.
    
    Returns:
        dict: A dictionary of the environment status
    """
    status = {}

    # Check if BeautifulSoup is installed, if not, log a warning and set status to False
    try:
        from bs4 import BeautifulSoup 
        status["BeautifulSoup installed"] = True
        logging.info(f"BeautifulSoup installed: {status['BeautifulSoup installed']}")
    except ImportError:
        status["BeautifulSoup installed"] = False
        logging.warning(f"BeautifulSoup installed: {status['BeautifulSoup installed']}")
    
    # Check if the BASE_URL is reachable, if not, log a warning and set status to False
    try:
        r = requests.get(BASE_URL, headers=HEADER, timeout=10)
        status[f"{BASE_URL} reachable"] = r.status_code == 200
        logging.info(f"{BASE_URL} reachable: {status[f'{BASE_URL} reachable']}")
    except requests.RequestException:
        status[f"{BASE_URL} reachable"] = False
        logging.warning(f"{BASE_URL} reachable: {status[f'{BASE_URL} reachable']}")
    
    # Check if the cache file exists, if not, log a warning and set status to False
    status["cached posts available"] = CACHE_PATH.exists()
    if status["cached posts available"]:
        logging.info(f"Cached posts available: {status['cached posts available']}")
    else:
        logging.warning(f"Cached posts available: {status['cached posts available']}")
        
    # Add spacey and download the English model if not already downloaded

    # Check if the WordNet lemmatiser is available, if not, log a warning and set status to False
    # try:
    #     import nltk
    #     nltk.download("wordnet", quiet=True)
    #     nltk.download("omw-1.4", quiet=True)
    #     from nltk.stem import WordNetLemmatizer
    #     WordNetLemmatizer().lemmatize("models")
    #     status["WordNet lemmatiser"] = True
    #     logging.info(f"WordNet lemmatiser available: {status['WordNet lemmatiser']}")
    # except Exception:
    #     status["WordNet lemmatiser"] = False
    #     logging.warning(f"WordNet lemmatiser available: {status['WordNet lemmatiser']}")    
    
    # If there is no network access and no cache, log a critical message and raise an error
    if not (status[f"{BASE_URL} reachable"] or status["cached posts available"]):
        # throw an error if there is no network access and no cache
        logging.critical("\n⚠️  No network access and no cache. Source the cached data file "
                      f"and place it at {CACHE_PATH}.") 
        raise RuntimeError("No network access and no cache available.")
        
    return status

ENV = preflight()
HAVE_BEATIFULSOUP = ENV["BeautifulSoup installed"]
HAVE_CACHE = ENV["cached posts available"]
ONLINE = ENV[f"{BASE_URL} reachable"]
# HAVE_LEMMATISER = ENV["WordNet lemmatiser"]

## 2. Web Scraping 

Create RepectfulScraper <br>
Source: BPP University (2026) ‘Topic 3: Web Scraping.ipynb’, Unstructured Data. Available at: BPP (Accessed: 30 October 2026).

Scrape forum for x threads urls
Scrape those thread urls to get first post in the thread


In [ ]:
# First we should create a ethical scraper which respects the robots.txt file and the terms of service of the website.

class RespectfulScraper:
    """
    A web scraper that respects robots.txt rules and rate limits.

    This class automatically:
    - Parses robots.txt for the target domain
    - Enforces crawl delays between requests
    - Identifies itself with a descriptive User-Agent string
    - Logs all activity for transparency and debugging
    """

    def __init__(self, base_url: str, header: dict, default_delay: int = 3):
        self.base_url = base_url.rstrip("/")

        # Create a session for connection reuse and consistent headers
        self.session = requests.Session()
        self.session.headers.update(header)

       # Set up logging
        self.logger = logging.getLogger(f"scraper.{urlparse(base_url).hostname}")
        self.logger.setLevel(logging.INFO)

        # Parse robots.txt
        robots_url = f"{self.base_url}/robots.txt"
        self.rp = RobotFileParser()
        self.rp.set_url(robots_url)

        resp = self.session.get(robots_url, timeout=10)
        
        if resp.status_code == 200:
            self.rp.parse(resp.text.splitlines())
            self.logger.info(f"Response {resp.status_code}: Successfully parsed robots.txt")
        elif resp.status_code in (401, 403):
            self.rp.disallow_all = True
            self.logger.warning(f"robots.txt returned {resp.status_code}; treating site as off-limits")
        elif 400 <= resp.status_code < 500:
            self.rp.allow_all = True
            self.logger.info(f"No robots.txt ({resp.status_code}); everything allowed")
        else:
            raise RuntimeError(f"Couldn't fetch robots.txt ({resp.status_code}); not crawling")

        delay = self.rp.crawl_delay(self.session.headers["User-Agent"])
        self.crawl_delay = float(delay) if delay else default_delay

    def can_fetch(self, url: str) -> bool:
        """Check if a URL is allowed by robots.txt."""
        return self.rp.can_fetch(self.session.headers["User-Agent"], url)

    def get(self, url: str) -> Optional[BeautifulSoup]:
        """
        Fetch and parse a page, respecting robots.txt and crawl delays.

        Returns a BeautifulSoup object if successful, None otherwise.
        """
        if not self.can_fetch(url):
            self.logger.warning(f"Blocked by robots.txt: {url}")
            return None

        try:
            self.logger.info(f"Fetching {url}")
            response = self.session.get(url, timeout=10)
            response.raise_for_status()

            # Respect crawl delay
            time.sleep(self.crawl_delay)

            return BeautifulSoup(response.text, "html.parser")

        except requests.exceptions.RequestException as e:
            self.logger.error(f"Error fetching {url}: {e}")
            return None

In [ ]:
# Create a respectful scraper for the MoneySavingExpert forums
mse_scraper = RespectfulScraper(base_url=BASE_URL, default_delay=DELAY, header=HEADER)

In [ ]:
# Function to check if the scraper can fetch the forum, search, and sitemap pages
def check_scraper_access(scraper: RespectfulScraper):
    """Check if the scraper can ethically fetch the urls.

    Args:
        scraper (RespectfulScraper): An instance of the RespectfulScraper class to check access for.
        
    Returns:
        None: Logs the access status for each URL.
    """
    urls_to_check = [FORUM_URL, SEARCH_URL, SITEMAP_URL]

    for url in urls_to_check:
        can_fetch = scraper.can_fetch(url)
        logging.info(f"Can fetch {url}: {can_fetch}")

check_scraper_access(mse_scraper)

In [ ]:

import re

def extract_thread_id(url: str | None) -> int | None:
    """Return the numeric thread ID from a discussion URL, or None if absent."""
    if not url:
        return None
    match = re.compile(r"/discussion/(\d+)").search(url)
    return int(match.group(1)) if match else None


In [ ]:
# Create a comparison between the 2 scrpaing options: search and sitemap. 
# Number of pages scraped
# Length of time taken 

### Function to scrape navigation menu for pages, get urls, and then scrpae threads

In [ ]:
# 1. Create scraper and check the robots.txt file etc
# 2. Scrpae the url provided and get the thread data (url, pinned, announcement, countViews, countComments)
    ## Count the number of records if less than the number specified in the config, then scrape the next page until the number of records is reached or there are no more pages to scrape.
    
def scrape_forum(scraper, forum_url: str, max_threads: int = 10) -> pd.DataFrame:
    """Scrape a forum for the specified number of threads and return a DataFrame with the thread data.

    Args:
        scraper (_type_): _description_
        forum_url (str): _description_
        max_threads (int, optional): _description_. Defaults to 10.

    Returns:
        pd.DataFrame: _description_
    """
    
    count_of_threads = 0
    
    # Extract the number of posts from the current page
    thread_data = []
    
    url_list = [forum_url]
    
    for url in url_list:
        if count_of_threads >= max_threads:
            break
        
        else:
            soup = scraper.get(url)
            for dataprops in soup.find_all("div", attrs={"data-props": True}):
                props = json.loads(dataprops["data-props"])

                for thread in props.get("discussions", []):
                    if count_of_threads >= max_threads:
                        break
                    else:
                        if thread.get("pinned") is not True and thread.get("announce") is not True:
                            thread_data.append({
                                "url": thread.get("url"),
                                "thread_id": extract_thread_id(thread.get("url")),
                                "pinned": thread.get("pinned"),
                                "announcement": thread.get("announce"),
                                "count_views": thread.get("countViews"),
                                "count_comments": thread.get("countComments"),
                            })
                            count_of_threads += 1
            if count_of_threads >= max_threads:
                break
            else: 
                next_page_link = soup.find("div", class_="PageControls Top").find("a", class_="Next")["href"]
                url_list.append(next_page_link)
    
    logging.info(f"Scraped {len(thread_data)} threads")
                
    return thread_data
    

In [ ]:
threads = scrape_forum(mse_scraper, FORUM_URL, max_threads=NUMBER_OF_THREADS_TO_SCRAPE)

In [ ]:
threads

In [ ]:
def scrape_first_posts(scraper, threads):
    scraped_posts = []

    for thread in threads:
        thread = thread.copy()
        soup = scraper.get(thread.get("url"))

        thread.update({
            "title": soup.find("div", class_="PageTitle pageHeadingBox isLarge").get_text(strip=True),
            "first_post_content": soup.find("div", class_="Message userContent").get_text(" ", strip=True),
            "first_post_author": soup.find("div", class_="Item-Header DiscussionHeader").find("a").get("title"),
            "first_post_date": soup.find("span", class_="MItem DateCreated").find("time").get("datetime"),
        })
        scraped_posts.append(thread)

    logging.info("Scraped %d posts", len(scraped_posts))
    
    return scraped_posts

In [ ]:
posts = scrape_threads(mse_scraper, threads)


In [ ]:
posts

In [ ]:
def pseudo(posts):
    for entry in posts:
        thread_id = str(entry.get("thread_id", ""))
        author = str(entry.get("first_post_author", ""))

        entry["thread_pseudo_id"] = hashlib.sha256(thread_id.encode("utf-8")).hexdigest()

        entry["author_pseudo_id"] = hashlib.sha256(author.encode("utf-8")).hexdigest()

    return posts

pseudo_posts = pseudo(posts)

In [ ]:
# create ficntioncalled mininmise_posts whihgc takes a list of dictuionnairies and removes url thread_id and author

def minimise_posts(posts):
    for entry in posts:
        entry.pop("url", None)
        entry.pop("thread_id", None)
        entry.pop("first_post_author", None)
    return posts

minimised_posts = minimise_posts(pseudo_posts)

minimised_posts

### XML scraper function to scrape sitemap.xml, gather urls and then scrpa each post

Sitemape urls are structured like:

https://forums.moneysavingexpert.com/sitemap-category-benefits-tax-credits-1001-1000.xml

where -1000.xml means there are 1000 threads per page
-1- and -1001- is where the list starts

In [ ]:
def sitemap_paignation(sitemap_url, max_threads: int = 10) -> list:
    threads_per_page = sitemap_url.split("-")[-1].split(".")[0]
    threads_per_page = int(threads_per_page)
    
    if max_threads <= threads_per_page:
        return [sitemap_url]
    else:
        num_pages = (max_threads // threads_per_page) + (1 if max_threads % threads_per_page > 0 else 0)
        # add starting url to the list of urls to scrape
        urls = [sitemap_url]
        urls += [sitemap_url.replace("-1-1000.xml", f"-{i}001-1000.xml") for i in range(1, num_pages)]
        return urls

In [ ]:
list_of_sitemaps = sitemap_paignation(SITEMAP_URL, max_threads=NUMBER_OF_THREADS_TO_SCRAPE)
list_of_sitemaps

In [ ]:
def get_thread_urls_from_sitemap(
    list_of_sitemaps: list,
    max_threads: int
) -> pd.DataFrame:
    """Get thread URLs sequentially until max_threads is reached."""

    thread_data = []
    ns = {"sm": "http://www.sitemaps.org/schemas/sitemap/0.9"}

    for sitemap_url in list_of_sitemaps:
        if len(thread_data) >= max_threads:
            break

        response = requests.get(sitemap_url, timeout=10)
        response.raise_for_status()

        root = ET.fromstring(response.content)

        for url_el in root.findall("sm:url", ns):
            if len(thread_data) >= max_threads:
                break

            loc = url_el.findtext("sm:loc", namespaces=ns)

            if loc:
                thread_data.append(
                    {"url": loc, 
                    "thread_id": extract_thread_id(loc)
                    }),

    logging.info(
        f"Scraped {len(thread_data)}/{max_threads} thread URLs from sitemap"
    )

    return thread_data

In [ ]:
threads = get_thread_urls_from_sitemap(list_of_sitemaps, max_threads=NUMBER_OF_THREADS_TO_SCRAPE)


In [ ]:
threads

In [ ]:
scraped_threads = scrape_threads(mse_scraper, threads)

In [ ]:
scraped_threads

In [ ]:

pseudo_xml_posts = pseudo(scraped_threads)

In [ ]:
minimised_xml_posts = minimise_posts(pseudo_xml_posts)
minimised_xml_posts

### Function to compare the two scraping methods for timing efficiency

In [ ]:
def test_time_for_both_scraping_methods():
    """Test the time taken for both scraping methods: forum and sitemap."""

    # Test forum scraping
    start_time = time.time()
    threads_df_forum = scrape_forum(mse_scraper, FORUM_URL, max_threads=NUMBER_OF_THREADS_TO_SCRAPE)
    posts = scrape_threads(mse_scraper, threads_df_forum)
    end_time = time.time()
    forum_time = end_time - start_time
    logging.info(f"Forum scraping took {forum_time:.2f} seconds")

    # Test sitemap scraping
    start_time = time.time()
    list_of_sitemaps = sitemap_paignation(SITEMAP_URL, max_threads=NUMBER_OF_THREADS_TO_SCRAPE)
    threads_df_sitemap = get_thread_urls_from_sitemap(list_of_sitemaps, max_threads=NUMBER_OF_THREADS_TO_SCRAPE)
    scraped_threads_sitemap = scrape_threads(mse_scraper, threads_df_sitemap)
    end_time = time.time()
    sitemap_time = end_time - start_time
    logging.info(f"Sitemap scraping took {sitemap_time:.2f} seconds")

    return forum_time, sitemap_time

In [ ]:
test_time_for_both_scraping_methods()

## EXTRACT FUNCTIONS

These functions make up the extrcation part of the pipeline

LOAD CACHE -> SCRAPE -> COLLECT -> SAVE TO CACHE

In [ ]:
def load_cache(cache_path: Path, refresh_cache: bool) -> tuple[list[dict], bool]:
    """Load JSONL cache entries and determine whether scraping is required."""
    if cache_path.exists() and not refresh_cache:
        with cache_path.open("r", encoding="utf-8") as cache_file:
            posts = [json.loads(line) for line in cache_file if line.strip()]

        logger.info("Loaded %d posts from %s", len(posts), cache_path)
        return posts, False

    if cache_path.exists():
        logger.info("Cache file exists but refresh is turned on.")
    else:
        logger.info("Cache file does not exist. Scraping required.")

    return [], True

In [ ]:
def scrape_forum(scraper, forum_url: str, max_threads: int = 10) -> pd.DataFrame:
    """Scrape a forum for the specified number of threads and return a DataFrame with the thread data.

    Args:
        scraper (_type_): _description_
        forum_url (str): _description_
        max_threads (int, optional): _description_. Defaults to 10.

    Returns:
        pd.DataFrame: _description_
    """
    
    count_of_threads = 0
    
    # Extract the number of posts from the current page
    thread_data = []
    
    url_list = [forum_url]
    
    for url in url_list:
        if count_of_threads >= max_threads:
            break
        
        else:
            soup = scraper.get(url)
            for dataprops in soup.find_all("div", attrs={"data-props": True}):
                props = json.loads(dataprops["data-props"])

                for thread in props.get("discussions", []):
                    if count_of_threads >= max_threads:
                        break
                    else:
                        if thread.get("pinned") is not True and thread.get("announce") is not True:
                            thread_data.append({
                                "url": thread.get("url"),
                                "thread_id": extract_thread_id(thread.get("url")),
                                "pinned": thread.get("pinned"),
                                "announcement": thread.get("announce"),
                                "count_views": thread.get("countViews"),
                                "count_comments": thread.get("countComments"),
                            })
                            count_of_threads += 1
            if count_of_threads >= max_threads:
                break
            else: 
                next_page_link = soup.find("div", class_="PageControls Top").find("a", class_="Next")["href"]
                url_list.append(next_page_link)
    
    logging.info(f"Scraped {len(thread_data)} threads")
                
    return thread_data
    
def scrape_first_posts(scraper, threads):
    scraped_posts = []

    for thread in threads:
        thread = thread.copy()
        soup = scraper.get(thread.get("url"))

        thread.update({
            "title": soup.find("div", class_="PageTitle pageHeadingBox isLarge").get_text(strip=True),
            "first_post_content": soup.find("div", class_="Message userContent").get_text(" ", strip=True),
            "first_post_author": soup.find("div", class_="Item-Header DiscussionHeader").find("a").get("title"),
            "first_post_date": soup.find("span", class_="MItem DateCreated").find("time").get("datetime"),
        })
        scraped_posts.append(thread)

    logging.info("Scraped %d posts", len(scraped_posts))
    
    return scraped_posts

def perform_scraping(scraper, forum_url, max_threads) -> list[dict]:
    """Scrape forum threads and return their first posts."""
    threads = scrape_forum(scraper, forum_url, max_threads)
    posts = scrape_first_posts(scraper, threads)
    logger.info("Scraped %d posts from the forum", len(posts))
    return posts
    

In [ ]:
SALT = os.environ.get("PSEUDONYM_SALT") or secrets.token_hex(16)  # never saved to disk

def pseudo_post(post):
    thread_id = str(post.get("thread_id", ""))
    author = str(post.get("first_post_author", ""))

    post["thread_pseudo_id"] = hashlib.sha256(
        f"{SALT}:{thread_id}".encode("utf-8")
    ).hexdigest()[:6]  
    post["author_pseudo_id"] = hashlib.sha256(
        f"{SALT}:{author}".encode("utf-8")
    ).hexdigest()[:6]

    return post


def minimise_post(post):
    post.pop("url", None)
    post.pop("thread_id", None)
    post.pop("first_post_author", None)
    return post





In [ ]:
def remove_pii(text: str) -> str:
    """Replace detected person names, email addresses, and UK phone numbers."""
    nlp = spacy.load("en_core_web_sm")

    # Mask detected names, working backwards to preserve character positions.
    doc = nlp(text)
    for entity in reversed(doc.ents):
        if entity.label_ == "PERSON":
            text = text[:entity.start_char] + "[NAME]" + text[entity.end_char:]

    text = re.sub(
        r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b",
        "[EMAIL]",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(
        r"(?<!\w)(?:\+44\s?(?:\(0\)\s?)?|0)(?:\d[\s().-]?){8,10}\d(?!\w)",
        "[PHONE]",
        text,
    )

    return text

In [ ]:
# def extract_or_collect(cache_path, cache_meta, refresh_cache, scraper, forum_url, max_threads):
#     posts, scrape_required = load_cache(cache_path, refresh_cache)
#     if scrape_required:
#         scraped_posts = perform_scraping(scraper, forum_url, max_threads)
#         for post in scraped_posts:
#             posts.append(post)
            
#     cache_path.parent.mkdir(parents=True, exist_ok=True)
    
#     count_of_cached_posts = 0
    
#     with cache_path.open("w", encoding="utf-8") as f:
#         for post in posts:
#             post = pseudo_post(post)
#             post = minimise_post(post)
#             if not any(p["thread_pseudo_id"] == post["thread_pseudo_id"] for p in posts):
#                 f.write(json.dumps(post, ensure_ascii=False) + "\n")
#                 count_of_cached_posts += 1
                
#     meta = {"collected_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
#         "num_posts": count_of_cached_posts,
#         "forum": forum_url}
#     cache_meta.write_text(json.dumps(meta, indent=2))
#     return posts, meta

In [ ]:
def extract_or_collect(cache_path, cache_meta, refresh_cache, scraper, forum_url, max_threads):
    posts, scrape_required = load_cache(cache_path, refresh_cache)   # cached posts are already cleaned
    if not scrape_required:
        return posts, json.loads(cache_meta.read_text())

    seen_ids = {p["thread_pseudo_id"] for p in posts}
            
    for post in perform_scraping(scraper, forum_url, max_threads):
        post = minimise_post(pseudo_post(post))
        if post["thread_pseudo_id"] in seen_ids:
            continue
        post["first_post_content"] = remove_pii(post["first_post_content"])
        seen_ids.add(post["thread_pseudo_id"])
        posts.append(post)

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with cache_path.open("w", encoding="utf-8") as f:
        for post in posts:
            f.write(json.dumps(post, ensure_ascii=False) + "\n")

    meta = {
        "collected_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "num_posts": len(posts),
        "forum": forum_url,
    }
    cache_meta.write_text(json.dumps(meta, indent=2))
    return posts, meta

In [ ]:
posts, meta = extract_or_collect(
    cache_path=CACHE_PATH, cache_meta=CACHE_META, refresh_cache=True, scraper=mse_scraper, forum_url=FORUM_URL, max_threads=5)

In [ ]:
posts

In [ ]:
meta

### TRANSFORMATION 1 FUCNTIONS

Now the data is stored in JSON, we need to do some initial tranformations before loading. This includes anything to be done on teh raw, including sentiment aanyalsis. 
Other erichment includes calculating the average post views per day - may give us a popularity rating
Could no some NLP to list GOVERNMENT DEPTS, BENEFIT LINES / TAX CREDITS 
We could later analyse which ones appear toegther most often?
Breakdown topics into specific areas of issues?? -< RESEARCH HOW TO DO THIS!
Enforce a common schema

### Entire Pipeline

In [ ]:
def pipeline(cache_path, refresh_cache, scraper, forum_url, max_threads):
    posts = extract_or_collect(cache_path, refresh_cache, scraper, forum_url, max_threads)
    pseudo_posts = pseudo(posts)
    minimised_posts = minimise_posts(pseudo_posts)
    return minimised_posts

In [ ]:
min = pipeline(cache_path=CACHE_PATH, refresh_cache=REFRESH_CACHE, scraper=mse_scraper, forum_url=FORUM_URL, max_threads=NUMBER_OF_THREADS_TO_SCRAPE)